# Table of Contents
* [Section 1: Imports](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)

---
## Section 1: Imports <a id="imports"></a>
---

In [1]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import extract_rhyme_unit, extract_end_words
from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD, 
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html

from praatio import textgrid

C:\Users\georg\anaconda3\envs\mfa_env\python.exe


---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [2]:
tg = textgrid.openTextgrid("output/twinkle.TextGrid", includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

Interval(start=1.89, end=3.12, label='twinkle')
Interval(start=3.12, end=4.23, label='twinkle')
Interval(start=4.23, end=5.05, label='little')
Interval(start=5.05, end=5.85, label='star')
Interval(start=6.18, end=6.64, label='how')
Interval(start=6.64, end=6.67, label='i')
Interval(start=6.67, end=7.1, label='wonder')
Interval(start=7.1, end=8.66, label='what')
Interval(start=8.85, end=9.48, label='you')
Interval(start=9.63, end=10.22, label='are')
Interval(start=10.94, end=11.67, label='sky')
Interval(start=12.29, end=12.94, label='high')
Interval(start=13.6, end=14.51, label='skillet')
Interval(start=15.14, end=15.81, label='skill')
Interval(start=16.15, end=16.6, label='at')


---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [3]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

twinkle: ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L']
twinkle: ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L']
little: ['L', 'IH1', 'T', 'AH0', 'L']
star: ['S', 'T', 'AA1', 'R']
how: ['HH', 'AW1']
i: ['AY1']
wonder: ['W', 'AH1', 'N', 'D', 'ER0']
what: ['HH', 'W', 'AH1', 'T']
you: ['Y', 'UW1']
are: ['AA1', 'R']
sky: ['S', 'K', 'AY1']
high: ['HH', 'AY1']
skillet: ['S', 'K', 'IH1', 'L', 'AH0', 'T']
skill: ['S', 'K', 'IH1', 'L']
at: ['AE1', 'T']
{'twinkle': ['T', 'W', 'IH1', 'NG', 'K', 'AH0', 'L'], 'little': ['L', 'IH1', 'T', 'AH0', 'L'], 'star': ['S', 'T', 'AA1', 'R'], 'how': ['HH', 'AW1'], 'i': ['AY1'], 'wonder': ['W', 'AH1', 'N', 'D', 'ER0'], 'what': ['HH', 'W', 'AH1', 'T'], 'you': ['Y', 'UW1'], 'are': ['AA1', 'R'], 'sky': ['S', 'K', 'AY1'], 'high': ['HH', 'AY1'], 'skillet': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'skill': ['S', 'K', 'IH1', 'L'], 'at': ['AE1', 'T']}


---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [4]:
rhyme_candidates = extract_end_words("input/twinkle.txt", word_to_phonemes)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'Twinkle twinkle little star', 'end_word': 'star', 'phonemes': ['S', 'T', 'AA1', 'R'], 'rhyme_unit': ['AA1', 'R']}
{'line_index': 1, 'line_text': 'How I wonder what you are', 'end_word': 'are', 'phonemes': ['AA1', 'R'], 'rhyme_unit': ['AA1', 'R']}
{'line_index': 2, 'line_text': 'sky', 'end_word': 'sky', 'phonemes': ['S', 'K', 'AY1'], 'rhyme_unit': ['AY1']}
{'line_index': 3, 'line_text': 'high', 'end_word': 'high', 'phonemes': ['HH', 'AY1'], 'rhyme_unit': ['AY1']}
{'line_index': 4, 'line_text': 'skillet', 'end_word': 'skillet', 'phonemes': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'rhyme_unit': ['IH1', 'L', 'AH0', 'T']}
{'line_index': 5, 'line_text': 'skill at', 'end_word': 'at', 'phonemes': ['AE1', 'T'], 'rhyme_unit': ['AE1', 'T']}


---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [5]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

{'wordA': 'star', 'wordB': 'are', 'similarity': 0.9999999999999999}
{'wordA': 'star', 'wordB': 'sky', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'high', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'skillet', 'similarity': 0.05}
{'wordA': 'star', 'wordB': 'at', 'similarity': 0.1}
{'wordA': 'are', 'wordB': 'sky', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'high', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'skillet', 'similarity': 0.05}
{'wordA': 'are', 'wordB': 'at', 'similarity': 0.1}
{'wordA': 'sky', 'wordB': 'high', 'similarity': 0.9999999999999999}
{'wordA': 'sky', 'wordB': 'skillet', 'similarity': 0.025}
{'wordA': 'sky', 'wordB': 'at', 'similarity': 0.05}
{'wordA': 'high', 'wordB': 'skillet', 'similarity': 0.025}
{'wordA': 'high', 'wordB': 'at', 'similarity': 0.05}
{'wordA': 'skillet', 'wordB': 'at', 'similarity': 0.15}
star {'star': 1.0, 'are': 0.9999999999999999, 'sky': 0.05, 'high': 0.05, 'skillet': 0.05, 'at': 0.1}
are {'star': 0.9999999999999999, 'are': 1.0, 'sky':

---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [6]:
# Clustering by thresholds from the similarity matrix
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=SIMILARITY_THRESHOLD) # SIMILARITY_THRESHOLD defined in 'clustering.py'

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "star": "A",
    "are": "A",
    "sky": "B",
    "high": "B",
    "skillet": "C",
    "at": "D"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_exact_rhymes(rhyme_candidates)

print(clusters)

# Debugging the clustering
clusters = cluster_exact_rhymes(rhyme_candidates)

debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

Clusters:
{'A': ['star', 'are'], 'B': ['sky', 'high'], 'C': ['skillet'], 'D': ['at']}

Word → Cluster Mapping:
{'star': 'A', 'are': 'A', 'sky': 'B', 'high': 'B', 'skillet': 'C', 'at': 'D'}
{'line_index': 0, 'line_text': 'Twinkle twinkle little star', 'end_word': 'star', 'phonemes': ['S', 'T', 'AA1', 'R'], 'rhyme_unit': ['AA1', 'R'], 'expected_cluster': 'A'}
{'line_index': 1, 'line_text': 'How I wonder what you are', 'end_word': 'are', 'phonemes': ['AA1', 'R'], 'rhyme_unit': ['AA1', 'R'], 'expected_cluster': 'A'}
{'line_index': 2, 'line_text': 'sky', 'end_word': 'sky', 'phonemes': ['S', 'K', 'AY1'], 'rhyme_unit': ['AY1'], 'expected_cluster': 'B'}
{'line_index': 3, 'line_text': 'high', 'end_word': 'high', 'phonemes': ['HH', 'AY1'], 'rhyme_unit': ['AY1'], 'expected_cluster': 'B'}
{'line_index': 4, 'line_text': 'skillet', 'end_word': 'skillet', 'phonemes': ['S', 'K', 'IH1', 'L', 'AH0', 'T'], 'rhyme_unit': ['IH1', 'L', 'AH0', 'T'], 'expected_cluster': 'C'}
{'line_index': 5, 'line_text': 'sk

---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [7]:
clusters = cluster_exact_rhymes(rhyme_candidates)

generate_rhyme_html(rhyme_candidates, clusters)

HTML file generated: generated_html/rhyme_visualization.html
